In [2]:
import numpy as np
import pandas as pd

np.random.seed(212)
n = 3000
menu_list = ["Kopi Susu", "Americano", "Cappuccino", "Roti Bakar", "Kentang Goreng", "Es Teh"]
cabang_list = ["Magelang", "Yogyakarta", "Semarang"]
waktu_list = ["Pagi", "Siang", "Sore", "Malam"]

data_tugas12 = pd.DataFrame({
    "nama_menu": np.random.choice(menu_list, size=n),
    "cabang": np.random.choice(cabang_list, size=n),
    "waktu_pesan": np.random.choice(waktu_list, size=n),
    "qty": np.random.randint(1, 6, size=n),
    "waktu_tunggu_menit": np.random.randint(2, 25, size=n),
})
    
# Simulasi: waktu tunggu lebih lama cenderung (namun tidak selalu) menurunkan peluang ulasan positif
prob_positif = 1 / (1 + np.exp((data_tugas12["waktu_tunggu_menit"] - 12) / 4))
data_tugas12["ulasan_positif"] = (np.random.rand(n) < prob_positif).astype(int)

data_tugas12.to_csv("tugas12_ulasan.csv", index=False)
print(f"Dataset berhasil dibuat: {len(data_tugas12)} baris")
print("Distribusi label:")
print(data_tugas12["ulasan_positif"].value_counts())

Dataset berhasil dibuat: 3000 baris
Distribusi label:
ulasan_positif
0    1610
1    1390
Name: count, dtype: int64


**A. Membangun ML Pipeline** *(bobot 25%)*

Buat `Pipeline` dengan tahapan: `StringIndexer` untuk `nama_menu`, `cabang`, `waktu_pesan` → `OneHotEncoder` untuk ketiganya → `VectorAssembler` (gabungkan hasil one-hot encoding dengan `qty` dan `waktu_tunggu_menit`) → `LogisticRegression` dengan `labelCol="ulasan_positif"`.

In [3]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Tugas12") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

df = spark.read.csv("file:///home/plont/Documents/Praktikum_Big_Data/tugas/tugas_12/tugas12_ulasan.csv", header=True,inferSchema=True)
df.show()

26/11/26 15:10:35 WARN Utils: Your hostname, localhost resolves to a loopback address: 127.0.0.1; using 10.102.250.151 instead (on interface wlo1)
26/11/26 15:10:35 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/26 15:10:35 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


+--------------+----------+-----------+---+------------------+--------------+
|     nama_menu|    cabang|waktu_pesan|qty|waktu_tunggu_menit|ulasan_positif|
+--------------+----------+-----------+---+------------------+--------------+
|     Kopi Susu|Yogyakarta|       Sore|  5|                 5|             1|
|        Es Teh|Yogyakarta|       Pagi|  1|                21|             0|
|    Cappuccino|Yogyakarta|       Sore|  5|                15|             0|
|Kentang Goreng|  Semarang|      Malam|  2|                 5|             1|
|     Americano|Yogyakarta|      Malam|  2|                10|             0|
|     Americano|Yogyakarta|       Pagi|  3|                10|             1|
|     Americano|Yogyakarta|      Siang|  2|                 5|             1|
|Kentang Goreng|Yogyakarta|      Malam|  5|                16|             0|
|    Roti Bakar|  Semarang|       Pagi|  5|                21|             0|
|     Kopi Susu|  Semarang|       Sore|  1|                23|  

In [4]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml import Pipeline

idx_nama_menu = StringIndexer(inputCol="nama_menu", outputCol="nama_menu_idx")
idx_cabang = StringIndexer(inputCol="cabang", outputCol="cabang_idx")
idx_waktu_pesan = StringIndexer(inputCol="waktu_pesan", outputCol="waktu_pesan_idx")

ohe = OneHotEncoder(
    inputCols = ["nama_menu_idx", "cabang_idx", "waktu_pesan_idx"],
    outputCols = ["nama_menu_vec", "cabang_vec", "waktu_pesan_vec"]
)

assembler = VectorAssembler(
    inputCols = ["nama_menu_vec", "cabang_vec", "waktu_pesan_vec", "qty", "waktu_tunggu_menit"],
    outputCol = "features"
)

lr = LogisticRegression(featuresCol="features", labelCol="ulasan_positif")


**B. Pelatihan & Evaluasi** *(bobot 25%)*

Bagi data 80:20 (`seed=77`), latih pipeline, lalu evaluasi pada data testing menggunakan **accuracy**, **F1-score**, dan **AUC-ROC**. Tampilkan juga *confusion matrix* sederhana menggunakan `groupBy`.

In [4]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator

pipeline = Pipeline(stages=[idx_nama_menu, idx_cabang, idx_waktu_pesan, ohe, assembler, lr])

train, test = df.randomSplit([0.8, 0.2], seed=77)

model = pipeline.fit(train)
predictions = model.transform(test)

ev_acc = MulticlassClassificationEvaluator(labelCol="ulasan_positif", predictionCol="prediction", metricName="accuracy")
accuracy = ev_acc.evaluate(predictions)

ev_f1 = MulticlassClassificationEvaluator(labelCol="ulasan_positif", predictionCol="prediction", metricName="f1")
f1 = ev_f1.evaluate(predictions)

ev_auc = BinaryClassificationEvaluator(labelCol="ulasan_positif", metricName="areaUnderROC")
auc = ev_auc.evaluate(predictions)

print(f"Accuracy: {accuracy:.4f}")
print(f"F1-score: {f1:.4f}")
print(f"AUC: {auc:.4f}")

predictions.groupBy("ulasan_positif", "prediction").count().orderBy("ulasan_positif", "prediction").show()


Accuracy: 0.7929
F1-score: 0.7926
AUC: 0.8650
+--------------+----------+-----+
|ulasan_positif|prediction|count|
+--------------+----------+-----+
|             0|       0.0|  257|
|             0|       1.0|   57|
|             1|       0.0|   65|
|             1|       1.0|  210|
+--------------+----------+-----+



**C. Model Pembanding** *(bobot 20%)*

Latih **`RandomForestClassifier`** (`numTrees=30`) menggunakan pipeline dengan lima tahap feature engineering yang identik dari bagian A (hanya tahap terakhir yang diganti). Bandingkan ketiga metrik evaluasinya dengan model Logistic Regression pada bagian B.

In [9]:
from pyspark.ml.classification import RandomForestClassifier
rf = RandomForestClassifier(featuresCol="features", labelCol="ulasan_positif", numTrees=30)

pipeline2 = Pipeline(stages=[idx_nama_menu, idx_cabang, idx_waktu_pesan, ohe, assembler, rf])

train2, test2 = df.randomSplit([0.8, 0.2], seed=77)

model2 = pipeline2.fit(train2)
predictions2 = model2.transform(test2)
ev_acc_2 = MulticlassClassificationEvaluator(labelCol="ulasan_positif", predictionCol="prediction", metricName="accuracy")
accuracy_2 = ev_acc_2.evaluate(predictions2)

ev_f1_2 = MulticlassClassificationEvaluator(labelCol="ulasan_positif", predictionCol="prediction", metricName="f1")
f1_2 = ev_f1_2.evaluate(predictions2)

ev_auc_2 = BinaryClassificationEvaluator(labelCol="ulasan_positif", metricName="areaUnderROC")
auc_2 = ev_auc_2.evaluate(predictions2)

print(f"Accuracy: {accuracy_2:.4f}")
print(f"F1-score: {f1_2:.4f}")
print(f"AUC: {auc_2:.4f}")

predictions2.groupBy("ulasan_positif", "prediction").count().orderBy("ulasan_positif", "prediction").show()


Accuracy: 0.7963
F1-score: 0.7957
AUC: 0.8447
+--------------+----------+-----+
|ulasan_positif|prediction|count|
+--------------+----------+-----+
|             0|       0.0|  262|
|             0|       1.0|   52|
|             1|       0.0|   68|
|             1|       1.0|  207|
+--------------+----------+-----+



Perbadingan antara Logistic Regression dan Random Forest :

akurasi dari algoritma Random Forest sedikit lebih baik dari Logistic Regresion, F1-score dari Random Forest sedikit lebih baik dari Logistic Regresion, AUC dari Random Forest sedikit lebih buruk dari Logistic Regresion

**D. Analisis Kesalahan Model** *(bobot 15%)*

Dari hasil prediksi model **terbaik** (pilih salah satu dari B atau C berdasarkan AUC tertinggi), tampilkan 10 baris di mana model **salah** memprediksi (`label != prediction`). Amati: apakah ada pola tertentu pada kesalahan tersebut (mis. mayoritas terjadi pada `waktu_tunggu_menit` yang berada di sekitar ambang batas tertentu)?


In [14]:
from pyspark.sql.functions import col

salah_prediksi = predictions.filter(col("ulasan_positif") != col("prediction"))

salah_prediksi.select("nama_menu", "cabang", "waktu_pesan", "qty", "waktu_tunggu_menit", "ulasan_positif", "prediction", "probability").show(10,truncate=False)

+---------+--------+-----------+---+------------------+--------------+----------+----------------------------------------+
|nama_menu|cabang  |waktu_pesan|qty|waktu_tunggu_menit|ulasan_positif|prediction|probability                             |
+---------+--------+-----------+---+------------------+--------------+----------+----------------------------------------+
|Americano|Magelang|Malam      |1  |15                |1             |0.0       |[0.728599672498492,0.27140032750150794] |
|Americano|Magelang|Siang      |2  |22                |1             |0.0       |[0.770344921639952,0.22965507836004792] |
|Americano|Magelang|Sore       |1  |7                 |0             |1.0       |[0.27022029368042216,0.7297797063195779]|
|Americano|Magelang|Sore       |2  |11                |0             |1.0       |[0.33594949241080563,0.6640505075891944]|
|Americano|Magelang|Sore       |3  |5                 |0             |1.0       |[0.26630856293948657,0.7336914370605135]|
|Americano|Magel

Machine learning tersebut terlihat sangat berpatokan pada waktu tunggu menit untuk memberi prediksi. Sehingga pada tabel tersebut terlihat seluruh prediksi salah. 
Kesalahan lain pada machine learning tersebut juga terlihat pada angka probabilitas yang secara pola selalu memiliki jarak antara 0 dan 1 sekitar 20% khususnya ketika menebak pada waktu tunggu yang mendekati batas 11 sampai 15  

**E. Kesimpulan Bisnis** *(bobot 15%)*

Tulis pada markdown cell (**minimal 100 kata**): berdasarkan model yang telah dibangun, apa rekomendasi konkret yang dapat diberikan kepada manajemen "Kopi Nusantara" untuk meningkatkan kepuasan pelanggan? Kaitkan dengan fitur yang paling berpengaruh terhadap prediksi (boleh merujuk pada pola yang ditemukan di bagian D).

Jawaban :

Dari prediksi machine learning tersebut menggunakan algoritma Logistic Regression, machine learning tersebut mayoritas salah menebak ulasan karena terlalu berpatokan pada waktu tunggu. Terlihat dari tabel tersebut, algoritma Logistic Regression langsung menebak ulasan negatif hanya karena waktu tunggu melebihi 12 menit. Namun, jika melihat pada machine learning yang menggunakan algoritma Random Forest dan menampilkan bagian yang sama, hasil dari tebakan algoritma random forest dinilai mirip dengan algoritma Logistic Regression. Kedua algoritma tersebut memang terlihat memiliki kesalahan yang sama, namun keduanya memiliki tingkat jumlah tebakan benar yang berbeda.
Dari informasi tersebut, ulasan positif terkadang dipengaruhi oleh berbagai hal bahkan termasuk psikologi pelanggan. Misalnya pelanggan yang sabar dan mudah menghargai orang lain cenderung memberi ulasan positif jika pelayanannya sesuai standar. Namun ada juga pelanggan yang memberikan ulasan negatif hanya karena masalah yang dinilai kecil dan tidak menganggu. 
Rekomendasi yang dapat diberikan jika terkait pelayanan adalah memberikan pelayanan sebaik mungkin dan selalu memenuhi standar yang ada.